# Part 1 — Data Cleaning & Engineering

In [1]:
!pip install gdown

import gdown

file_id = "1ikp3Dcsf2CvLf3pe7NyXwm9aimP4AoYO"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "data.csv", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1ikp3Dcsf2CvLf3pe7NyXwm9aimP4AoYO
From (redirected): https://drive.google.com/uc?id=1ikp3Dcsf2CvLf3pe7NyXwm9aimP4AoYO&confirm=t&uuid=5dd3b5a1-4fd0-46aa-9e61-2217ed2317bb
To: /kaggle/working/data.csv
100%|██████████| 241M/241M [00:03<00:00, 75.6MB/s] 


'data.csv'

a) Initialize a Spark session and load the CSV file into a Spark DataFrame.

In [2]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import math

spark = SparkSession.builder \
    .appName("CitiBike_Data_Cleaning_Engineering") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "12g") \
    .getOrCreate()

print("Spark session initialized")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 17:40:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session initialized
Spark version: 4.0.2


In [3]:
df = spark.read.csv(
    "data.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.printSchema()
df.limit(5).toPandas()

Dataset loaded successfully


26/05/23 17:40:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , starttime, stoptime, start station id, start station name, start station latitude, start station longitude, end station id, end station name, end station latitude, end station longitude, bikeid, usertype, birth year, gender
 Schema: _c0, starttime, stoptime, start station id, start station name, start station latitude, start station longitude, end station id, end station name, end station latitude, end station longitude, bikeid, usertype, birth year, gender
Expected: _c0 but found: 
CSV file: file:///kaggle/working/data.csv


Rows: 1,300,000
Columns: 15
root
 |-- _c0: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)



,_c0,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,0,2019-04-17 14:37:03.844,2019-04-17 14:43:13.767,264.0,Maiden Ln & Pearl St,40.707065,-74.007319,330.0,Reade St & Broadway,40.714505,-74.005628,16906,Subscriber,1969,1
1,1,2019-04-17 14:37:01.225,2019-04-17 14:42:48.108,411.0,E 6 St & Avenue D,40.722281,-73.976687,438.0,St Marks Pl & 1 Ave,40.727791,-73.985649,33738,Subscriber,1974,1
2,2,2019-04-17 14:37:06.936,2019-04-17 14:52:25.604,128.0,MacDougal St & Prince St,40.727103,-74.002971,358.0,Christopher St & Greenwich St,40.732916,-74.007114,27351,Subscriber,1969,2
3,3,2019-04-17 14:37:02.985,2019-04-17 14:46:53.331,349.0,Rivington St & Ridge St,40.718502,-73.983299,3435.0,Grand St & Elizabeth St,40.718822,-73.995960,19592,Subscriber,1986,1
4,4,2019-04-17 14:37:03.800,2019-04-17 14:42:34.710,3556.0,24 St & 41 Ave,40.752708,-73.939740,3563.0,28 St & 36 Ave,40.757186,-73.932719,35244,Subscriber,1979,2


b) Begin by exploring the dataset and identify any anomalies or inconsistencies. You are expected to
clean and prepare the data. Some values may be missing explicitly (e.g., NULL) or implicitly (e.g.,
zeroes or empty strings). Handle such cases thoughtfully in a way that allows for reliable querying
and analysis.

In [4]:
# Standardizing Column Names

# Checking for an Unnecessary Index Column, Spark usually names it _c0
if "_c0" in df.columns:
    df = df.drop("_c0")
    print(" Dropped unnecessary index column: _c0")

# Renaming columns to clean snake_case names
df = df \
    .withColumnRenamed("start station id",        "start_station_id") \
    .withColumnRenamed("start station name",      "start_station_name") \
    .withColumnRenamed("start station latitude",  "start_station_lat") \
    .withColumnRenamed("start station longitude", "start_station_longitude") \
    .withColumnRenamed("end station id",          "end_station_id") \
    .withColumnRenamed("end station name",        "end_station_name") \
    .withColumnRenamed("end station latitude",    "end_station_latitude") \
    .withColumnRenamed("end station longitude",   "end_station_lng") \
    .withColumnRenamed("bikeid",                  "bike_id") \
    .withColumnRenamed("usertype",                "user_type") \
    .withColumnRenamed("birth year",              "birth_year")

print(" Column names standardized")
df.printSchema()

 Dropped unnecessary index column: _c0
 Column names standardized
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start_station_id: double (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_lat: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_id: double (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_lng: double (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- user_type: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- gender: integer (nullable = true)



In [5]:
# Exploring Explicit Missing Values

print("Missing Values per Column:")
print("─" * 60)

total = df.count()
missing_data = []

for c in df.columns:
    missing_count = df.filter(
        F.col(c).isNull() | (F.col(c).cast("string") == "")
    ).count()
    missing_data.append((c, missing_count))

missing_data.sort(key=lambda x: -x[1])

for col_name, count in missing_data:
    pct = count / total * 100
    print(f"{col_name:<30}: {count:>6,} ({pct:.4f}%)")

print("─" * 60)

Missing Values per Column:
────────────────────────────────────────────────────────────


start_station_id              :     33 (0.0025%)
start_station_name            :     33 (0.0025%)
end_station_id                :     33 (0.0025%)
end_station_name              :     33 (0.0025%)
starttime                     :      0 (0.0000%)
stoptime                      :      0 (0.0000%)
start_station_lat             :      0 (0.0000%)
start_station_longitude       :      0 (0.0000%)
end_station_latitude          :      0 (0.0000%)
end_station_lng               :      0 (0.0000%)
bike_id                       :      0 (0.0000%)
user_type                     :      0 (0.0000%)
birth_year                    :      0 (0.0000%)
gender                        :      0 (0.0000%)
────────────────────────────────────────────────────────────


In [6]:
# Checking whether Station missing values occur together

station_cols = [
    "start_station_id",
    "start_station_name",
    "end_station_id",
    "end_station_name"
]

df_station_check = df.withColumn(
    "missing_station_fields_count",
    sum([
        F.when(
            F.col(c).isNull() | (F.col(c).cast("string") == ""),
            1
        ).otherwise(0)
        for c in station_cols
    ])
)

rows_with_any_station_missing = df_station_check.filter(
    F.col("missing_station_fields_count") > 0
).count()

rows_with_all_station_missing = df_station_check.filter(
    F.col("missing_station_fields_count") == len(station_cols)
).count()

print("Station Missingness Check")
print("─" * 60)
print(f"Rows with at least one missing station field : {rows_with_any_station_missing}")
print(f"Rows with all four station fields missing    : {rows_with_all_station_missing}")

if rows_with_any_station_missing == rows_with_all_station_missing:
    print("Missing station values occur in the same rows.")
else:
    print("Missing station values do not all occur in the same rows.")

df_station_check.filter(
    F.col("missing_station_fields_count") > 0
).select(station_cols + ["missing_station_fields_count"]).limit(20).toPandas()

Station Missingness Check
────────────────────────────────────────────────────────────
Rows with at least one missing station field : 33
Rows with all four station fields missing    : 33
Missing station values occur in the same rows.


,start_station_id,start_station_name,end_station_id,end_station_name,missing_station_fields_count
0,NaN,None,NaN,None,4
1,NaN,None,NaN,None,4
2,NaN,None,NaN,None,4
3,NaN,None,NaN,None,4
4,NaN,None,NaN,None,4
5,NaN,None,NaN,None,4
6,NaN,None,NaN,None,4
7,NaN,None,NaN,None,4
8,NaN,None,NaN,None,4
9,NaN,None,NaN,None,4


In [7]:
# Detecting Implicit Missing Values

print("Searching for Implicit Missing Values")
print("─" * 70)

total = df.count()

string_cols  = ["start_station_name", "end_station_name", "user_type"]
numeric_cols = ["start_station_id", "end_station_id", "birth_year"]
fake_missing_strings = ["null", "unknown", "none", ""]

print("\nString-based hidden missing values")
print("─" * 70)

for c in string_cols:
    print(f"\nColumn: {c}")
    found_any = False
    for val in fake_missing_strings:
        count = df.filter(
            F.lower(F.trim(F.col(c).cast("string"))) == val
        ).count()
        pct = count / total * 100
        print(f"  '{val if val != '' else 'empty string'}': {count:,} rows ({pct:.4f}%)")
        if count > 0:
            found_any = True
    if not found_any:
        print(" No hidden string missing values found")

print("\nNumeric suspicious values")
print("─" * 70)

for c in numeric_cols:
    zero_count = df.filter(F.col(c) == 0).count()
    pct = zero_count / total * 100
    print(f"{c:<25}: {zero_count:,} rows with value 0 ({pct:.4f}%)")

print("\nImportant note")
print("─" * 70)
print("gender = 0 is not missing because the dataset defines 0 as Unknown")

Searching for Implicit Missing Values
──────────────────────────────────────────────────────────────────────

String-based hidden missing values
──────────────────────────────────────────────────────────────────────

Column: start_station_name


  'null': 0 rows (0.0000%)


  'unknown': 0 rows (0.0000%)


  'none': 0 rows (0.0000%)


  'empty string': 0 rows (0.0000%)
 No hidden string missing values found

Column: end_station_name


  'null': 0 rows (0.0000%)


... (6 outputs left out)

  'empty string': 0 rows (0.0000%)
 No hidden string missing values found

Numeric suspicious values
──────────────────────────────────────────────────────────────────────


start_station_id         : 0 rows with value 0 (0.0000%)


end_station_id           : 0 rows with value 0 (0.0000%)


birth_year               : 0 rows with value 0 (0.0000%)

Important note
──────────────────────────────────────────────────────────────────────
gender = 0 is not missing because the dataset defines 0 as Unknown


No implicit missing values were detected in the previous check so we only handle explicit Spark null values.

In [8]:
# Handling Explicit Missing Values

# Creating a working copy of the dataframe
df_clean = df

print("Explicit missing values identified")
print("─" * 60)

important_cols = [
    "start_station_name",
    "end_station_name",
    "start_station_id",
    "end_station_id",
]

print("NULL counts before dropping station-missing rows:")
for c in important_cols:
    n = df_clean.filter(F.col(c).isNull()).count()
    print(f"{c:<30}: {n:>6,} NULLs")

rows_before = df_clean.count()
df_clean    = df_clean.dropna(subset=important_cols)
rows_after  = df_clean.count()

print("─" * 60)
print(f"Rows before dropping : {rows_before:,}")
print(f"Rows after dropping  : {rows_after:,}")
print(f"Dropped rows         : {rows_before - rows_after:,}")
print("─" * 60)

print("NULL counts after dropping station-missing rows:")
for c in important_cols:
    n = df_clean.filter(F.col(c).isNull()).count()
    print(f"{c:<30}: {n:>6,} NULLs")

Explicit missing values identified
────────────────────────────────────────────────────────────
NULL counts before dropping station-missing rows:


start_station_name            :     33 NULLs


end_station_name              :     33 NULLs


start_station_id              :     33 NULLs


end_station_id                :     33 NULLs


────────────────────────────────────────────────────────────
Rows before dropping : 1,300,000
Rows after dropping  : 1,299,967
Dropped rows         : 33
────────────────────────────────────────────────────────────
NULL counts after dropping station-missing rows:


start_station_name            :      0 NULLs


end_station_name              :      0 NULLs


start_station_id              :      0 NULLs


end_station_id                :      0 NULLs


**Cleaning decision:**
- No implicit missing values such as empty strings, 'NULL', 'unknown', 'none', or invalid zeroes were found.
- The dataset only contains explicit Spark NULL values.
- The 33 rows with missing station IDs/names were removed from df_clean because station information is essential for station-based analysis and cannot be imputed in this case.
- The original df is preserved unchanged for traceability.
- birth_year: 0 NULLs confirmed by output — no imputation needed.

c) You are expected to create new columns that provide useful insights:

In [9]:
# Rider Age: Derive an Age column using Year of Birth
# Rider age should represent the age at the time of the trip

DATA_YEAR = 2019  # Trips are from 2019, so we use 2019 instead of the current year

df_clean = df_clean.withColumn(
    "rider_age",
    F.lit(DATA_YEAR) - F.col("birth_year")
)

print("rider_age column added")

print("Preview of birth_year and rider_age:")
display(df_clean.select("birth_year", "rider_age").limit(10).toPandas())

print("Summary statistics for rider_age:")
display(df_clean.select("rider_age").describe().toPandas())

null_age_count    = df_clean.filter(F.col("rider_age").isNull()).count()
negative_age_count = df_clean.filter(F.col("rider_age") < 0).count()
invalid_age_count  = df_clean.filter(
    (F.col("rider_age") < 12) | (F.col("rider_age") > 100)
).count()

print("Validation results:")
print(f"NULL rider_age values     : {null_age_count:,}")
print(f"Negative rider_age values : {negative_age_count:,}")
print(f"Age noise candidates      : {invalid_age_count:,}")

rider_age column added
Preview of birth_year and rider_age:


,birth_year,rider_age
0,1969,50
1,1974,45
2,1969,50
3,1986,33
4,1979,40
5,1958,61
6,1977,42
7,1984,35
8,1996,23
9,1969,50


Summary statistics for rider_age:


,summary,rider_age
0,count,1299967
1,mean,38.99116977584816
2,stddev,12.16992886906696
3,min,16
4,max,145


Validation results:
NULL rider_age values     : 0
Negative rider_age values : 0
Age noise candidates      : 654


In [10]:
# Exploring age outliers — distribution of unrealistic birth years
df_clean.filter(
    (F.col("rider_age") < 12) | (F.col("rider_age") > 100)
).select("birth_year", "rider_age") \
 .groupBy("birth_year").count() \
 .orderBy("count", ascending=False) \
 .show(20)

+----------+-----+
|birth_year|count|
+----------+-----+
|      1900|  244|
|      1888|   76|
|      1890|   55|
|      1889|   47|
|      1901|   42|
|      1886|   24|
|      1896|   20|
|      1885|   20|
|      1899|   16|
|      1887|   16|
|      1893|   15|
|      1911|   13|
|      1912|   12|
|      1910|   10|
|      1917|    9|
|      1895|    8|
|      1907|    6|
|      1916|    5|
|      1915|    5|
|      1918|    4|
+----------+-----+
only showing top 20 rows


In [11]:
# Removing age outliers: rider_age > 100 is unrealistic and flagged as noise.
# Output from cell 15 confirmed min age = 16, so no records fall below 12.
# Filter applied here — before any engineered columns are built — so that
# all downstream DataFrames and the SQL view are constructed on clean data.

rows_before = df_clean.count()

df_clean = df_clean.filter(F.col("rider_age") <= 100)

rows_after   = df_clean.count()
rows_removed = rows_before - rows_after

print(f"Age outliers (rider_age > 100) removed.")
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Removed     : {rows_removed:,}")

Age outliers (rider_age > 100) removed.
Rows before : 1,299,967
Rows after  : 1,299,313
Removed     : 654


In [12]:
# Trip Duration: Calculate the duration in seconds

df_clean = df_clean.withColumn(
    "trip_duration_seconds",
    F.unix_timestamp("stoptime") - F.unix_timestamp("starttime")
)

print("trip_duration_seconds column added")

display(
    df_clean.select("starttime", "stoptime", "trip_duration_seconds")
            .limit(10).toPandas()
)

display(df_clean.select("trip_duration_seconds").describe().toPandas())

null_duration_count     = df_clean.filter(F.col("trip_duration_seconds").isNull()).count()
negative_duration_count = df_clean.filter(F.col("trip_duration_seconds") < 0).count()
zero_duration_count     = df_clean.filter(F.col("trip_duration_seconds") == 0).count()
short_duration_count    = df_clean.filter(F.col("trip_duration_seconds") < 60).count()

print("Validation results:")
print(f"NULL duration values      : {null_duration_count:,}")
print(f"Negative duration values  : {negative_duration_count:,}")
print(f"Zero duration values      : {zero_duration_count:,}")
print(f"Duration noise candidates : {short_duration_count:,}")

trip_duration_seconds column added


,starttime,stoptime,trip_duration_seconds
0,2019-04-17 14:37:03.844,2019-04-17 14:43:13.767,370
1,2019-04-17 14:37:01.225,2019-04-17 14:42:48.108,347
2,2019-04-17 14:37:06.936,2019-04-17 14:52:25.604,919
3,2019-04-17 14:37:02.985,2019-04-17 14:46:53.331,591
4,2019-04-17 14:37:03.800,2019-04-17 14:42:34.710,331
5,2019-04-17 14:37:03.811,2019-04-17 14:40:28.133,205
6,2019-04-17 14:37:06.711,2019-04-17 15:04:17.786,1631
7,2019-04-17 14:37:06.853,2019-04-17 14:57:18.467,1212
8,2019-04-17 14:37:06.936,2019-04-17 14:40:55.706,229
9,2019-04-17 14:37:08.497,2019-04-17 15:03:34.848,1586


,summary,trip_duration_seconds
0,count,1299313
1,mean,960.3161609250427
2,stddev,9521.48446412817
3,min,61
4,max,2943038


Validation results:
NULL duration values      : 0
Negative duration values  : 0
Zero duration values      : 0
Duration noise candidates : 0


In [13]:
# Trip Distance: Haversine formula using a Spark UDF

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculates straight-line distance between two lat/lon points in km.
    Returns None if any coordinate is missing.
    """
    if lat1 is None or lon1 is None or lat2 is None or lon2 is None:
        return None

    R = 6371.0  # Earth radius in km
    lat1_r = math.radians(lat1)
    lon1_r = math.radians(lon1)
    lat2_r = math.radians(lat2)
    lon2_r = math.radians(lon2)

    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r

    a = math.sin(dlat / 2)**2 + math.cos(lat1_r) * math.cos(lat2_r) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))

    return float(R * c)

haversine_udf = F.udf(haversine_distance, DoubleType())

df_clean = df_clean.withColumn(
    "trip_distance_km",
    haversine_udf(
        F.col("start_station_lat"),
        F.col("start_station_longitude"),
        F.col("end_station_latitude"),
        F.col("end_station_lng")
    )
)

print("trip_distance_km column added")

display(
    df_clean.select("start_station_name", "end_station_name", "trip_distance_km")
            .limit(10).toPandas()
)

display(df_clean.select("trip_distance_km").describe().toPandas())

null_distance_count     = df_clean.filter(F.col("trip_distance_km").isNull()).count()
negative_distance_count = df_clean.filter(F.col("trip_distance_km") < 0).count()
zero_distance_count     = df_clean.filter(F.col("trip_distance_km") == 0).count()

print("Validation results:")
print(f"NULL distance values     : {null_distance_count:,}")
print(f"Negative distance values : {negative_distance_count:,}")
print(f"Zero distance values     : {zero_distance_count:,}")

trip_distance_km column added


,start_station_name,end_station_name,trip_distance_km
0,Maiden Ln & Pearl St,Reade St & Broadway,0.839468
1,E 6 St & Avenue D,St Marks Pl & 1 Ave,0.972541
2,MacDougal St & Prince St,Christopher St & Greenwich St,0.734618
3,Rivington St & Ridge St,Grand St & Elizabeth St,1.067659
4,24 St & 41 Ave,28 St & 36 Ave,0.773090
5,Watts St & Greenwich St,Pier 40 - Hudson River Park,0.429543
6,Nassau St & Navy St,W 20 St & 11 Ave,5.839733
7,Fulton St & William St,E 6 St & Avenue B,2.664735
8,3 St & 3 Ave,7 St & 5 Ave,0.545859
9,Van Brunt St & Wolcott St,Smith St & 9 St,1.289965


,summary,trip_distance_km
0,count,1299313
1,mean,1.7748043171310999
2,stddev,1.410256809408169
3,min,0.0
4,max,16.732761099958836


Validation results:
NULL distance values     : 0
Negative distance values : 0
Zero distance values     : 27,697


In [14]:
# Validating Zero-Distance Trips
# Confirming that all zero-distance trips are round trips (same start and end station)
# rather than invalid or corrupt records.

zero_distance_df = df_clean.filter(F.col("trip_distance_km") == 0)

display(
    zero_distance_df.groupBy(
        (F.col("start_station_name") == F.col("end_station_name")).alias("same_station")
    ).count().toPandas()
)

display(
    zero_distance_df.select("start_station_name", "end_station_name", "trip_distance_km")
                    .limit(20).toPandas()
)

,same_station,count
0,True,27697


,start_station_name,end_station_name,trip_distance_km
0,1 Ave & E 68 St,1 Ave & E 68 St,0.0
1,5 Ave & E 73 St,5 Ave & E 73 St,0.0
2,Central Park West & W 72 St,Central Park West & W 72 St,0.0
3,5 Ave & E 73 St,5 Ave & E 73 St,0.0
4,Central Park West & W 72 St,Central Park West & W 72 St,0.0
5,Central Park West & W 72 St,Central Park West & W 72 St,0.0
6,Central Park West & W 72 St,Central Park West & W 72 St,0.0
7,5 Ave & E 78 St,5 Ave & E 78 St,0.0
8,Grand Army Plaza & Plaza St West,Grand Army Plaza & Plaza St West,0.0
9,Duffield St & Willoughby St,Duffield St & Willoughby St,0.0


In [15]:
# Trip Speed (km/h): distance / duration
# The spec requires a UDF for any unit conversions.
# A dedicated UDF handles the seconds-to-hours conversion.

def seconds_to_hours(seconds):
    """Converts trip duration from seconds to hours. Returns None for invalid input."""
    if seconds is None or seconds <= 0:
        return None
    return float(seconds / 3600.0)

seconds_to_hours_udf = F.udf(seconds_to_hours, DoubleType())

df_clean = df_clean.withColumn(
    "trip_speed_kmh",
    F.when(
        F.col("trip_duration_seconds") > 0,
        F.col("trip_distance_km") / seconds_to_hours_udf(F.col("trip_duration_seconds"))
    ).otherwise(None)
)

print("trip_speed_kmh column added")

display(
    df_clean.select("trip_distance_km", "trip_duration_seconds", "trip_speed_kmh")
            .limit(10).toPandas()
)

display(df_clean.select("trip_speed_kmh").describe().toPandas())

null_speed_count  = df_clean.filter(F.col("trip_speed_kmh").isNull()).count()
neg_speed_count   = df_clean.filter(F.col("trip_speed_kmh") < 0).count()
noise_speed_count = df_clean.filter(F.col("trip_speed_kmh") > 40).count()

print("Validation results:")
print(f"NULL speed values       : {null_speed_count:,}")
print(f"Negative speed values   : {neg_speed_count:,}")
print(f"Speed noise candidates  : {noise_speed_count:,}")

trip_speed_kmh column added


,trip_distance_km,trip_duration_seconds,trip_speed_kmh
0,0.839468,370,8.167793
1,0.972541,347,10.089763
2,0.734618,919,2.877720
3,1.067659,591,6.503508
4,0.773090,331,8.408227
5,0.429543,205,7.543203
6,5.839733,1631,12.889662
7,2.664735,1212,7.915053
8,0.545859,229,8.581184
9,1.289965,1586,2.928042


,summary,trip_speed_kmh
0,count,1299313
1,mean,8.768189045254575
2,stddev,3.3658596659677693
3,min,0.0
4,max,52.969812151630336


Validation results:
NULL speed values       : 0
Negative speed values   : 0
Speed noise candidates  : 1


In [16]:
# Time Categorization: Create a Period_of_Day feature

df_clean = df_clean.withColumn(
    "period_of_day",
    F.when((F.hour("starttime") >= 5)  & (F.hour("starttime") < 12), "Morning")
     .when((F.hour("starttime") >= 12) & (F.hour("starttime") < 17), "Afternoon")
     .when((F.hour("starttime") >= 17) & (F.hour("starttime") < 21), "Evening")
     .otherwise("Night")
)

print("period_of_day column added")

display(df_clean.groupBy("period_of_day").count().toPandas())

display(df_clean.select("starttime", "period_of_day").limit(10).toPandas())

null_period_count       = df_clean.filter(F.col("period_of_day").isNull()).count()
unexpected_period_count = df_clean.filter(
    ~F.col("period_of_day").isin("Morning", "Afternoon", "Evening", "Night")
).count()

print("Validation results:")
print(f"NULL period_of_day values    : {null_period_count:,}")
print(f"Unexpected period categories : {unexpected_period_count:,}")

period_of_day column added


,period_of_day,count
0,Evening,336781
1,Morning,433318
2,Afternoon,437863
3,Night,91351


,starttime,period_of_day
0,2019-04-17 14:37:03.844,Afternoon
1,2019-04-17 14:37:01.225,Afternoon
2,2019-04-17 14:37:06.936,Afternoon
3,2019-04-17 14:37:02.985,Afternoon
4,2019-04-17 14:37:03.800,Afternoon
5,2019-04-17 14:37:03.811,Afternoon
6,2019-04-17 14:37:06.711,Afternoon
7,2019-04-17 14:37:06.853,Afternoon
8,2019-04-17 14:37:06.936,Afternoon
9,2019-04-17 14:37:08.497,Afternoon


Validation results:
NULL period_of_day values    : 0
Unexpected period categories : 0


In [17]:
# Start Month: Extracted from the start timestamp

df_clean = df_clean.withColumn("start_month", F.month("starttime"))

print("start_month column added")

display(df_clean.groupBy("start_month").count().orderBy("start_month").toPandas())

display(df_clean.select("starttime", "start_month").limit(10).toPandas())

null_month_count    = df_clean.filter(F.col("start_month").isNull()).count()
invalid_month_count = df_clean.filter(
    (F.col("start_month") < 1) | (F.col("start_month") > 12)
).count()

print("Validation results:")
print(f"NULL start_month values    : {null_month_count:,}")
print(f"Invalid start_month values : {invalid_month_count:,}")

start_month column added


,start_month,count
0,1,49960
1,2,49966
2,3,99942
3,4,99944
4,5,99941
5,6,149932
6,7,149924
7,8,149924
8,9,149933
9,10,149928


,starttime,start_month
0,2019-04-17 14:37:03.844,4
1,2019-04-17 14:37:01.225,4
2,2019-04-17 14:37:06.936,4
3,2019-04-17 14:37:02.985,4
4,2019-04-17 14:37:03.800,4
5,2019-04-17 14:37:03.811,4
6,2019-04-17 14:37:06.711,4
7,2019-04-17 14:37:06.853,4
8,2019-04-17 14:37:06.936,4
9,2019-04-17 14:37:08.497,4


Validation results:
NULL start_month values    : 0
Invalid start_month values : 0


d) You are expected to identify and flag records that meet any of the following noise criteria:

In [18]:
# Identifying and Flagging Noisy Records
# Age outliers (> 100) were already removed in cell 16 before columns were engineered.
# This cell flags any remaining noise across all four criteria.

df_flagged = df_clean.withColumn(
    "noise_duration_lt_60",
    F.col("trip_duration_seconds") < 60
).withColumn(
    "noise_speed_gt_40",
    F.col("trip_speed_kmh") > 40
).withColumn(
    "noise_invalid_age",
    (F.col("rider_age") > 100) | (F.col("rider_age") < 12)
).withColumn(
    "noise_missing_identifiers",
    F.col("start_station_name").isNull() |
    F.col("end_station_name").isNull()   |
    F.col("bike_id").isNull()
)

df_flagged = df_flagged.withColumn(
    "is_noisy",
    F.col("noise_duration_lt_60")    |
    F.col("noise_speed_gt_40")       |
    F.col("noise_invalid_age")       |
    F.col("noise_missing_identifiers")
)

print("Noise flags added")

df_flagged.select(
    F.sum(F.col("noise_duration_lt_60").cast("int")).alias("duration_less_than_60"),
    F.sum(F.col("noise_speed_gt_40").cast("int")).alias("speed_greater_than_40"),
    F.sum(F.col("noise_invalid_age").cast("int")).alias("invalid_age"),
    F.sum(F.col("noise_missing_identifiers").cast("int")).alias("missing_identifiers"),
    F.sum(F.col("is_noisy").cast("int")).alias("total_noisy_rows")
).toPandas()

Noise flags added


,duration_less_than_60,speed_greater_than_40,invalid_age,missing_identifiers,total_noisy_rows
0,0,1,0,0,1


In [19]:
# Removing trips with speed > 40 km/h (unrealistic for a human-powered ride)

rows_before = df_clean.count()

df_clean = df_clean.filter(F.col("trip_speed_kmh") <= 40)

rows_after = df_clean.count()

print("Unrealistic speed records removed")
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Removed     : {rows_before - rows_after:,}")
print("Trips with speeds > 40 km/h were removed because they are unrealistic for "
      "normal Citi Bike rides and likely indicate system or recording errors.")

Unrealistic speed records removed
Rows before : 1,299,313
Rows after  : 1,299,312
Removed     : 1
Trips with speeds > 40 km/h were removed because they are unrealistic for normal Citi Bike rides and likely indicate system or recording errors.


In [20]:
# Part 2 — Analytical Queries using Spark SQL
# All filtering is now complete. The view is created from the fully cleaned DataFrame.
# This ensures every query runs on consistent, correct data.

df_clean.createOrReplaceTempView("citibike_clean")

print("Temporary SQL view 'citibike_clean' created successfully.")
print(f"Total rows in view: {df_clean.count():,}")

Temporary SQL view 'citibike_clean' created successfully.


Total rows in view: 1,299,312


In [21]:
# ============================================================
# Query 2(a): Percentage of round trips per user type
# ============================================================

# A round trip means the rider started and ended at the same station.

round_trips_by_user_type = spark.sql("""
SELECT
    user_type,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN start_station_name = end_station_name THEN 1 ELSE 0 END) AS round_trips,
    ROUND(
        SUM(CASE WHEN start_station_name = end_station_name THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS round_trip_percentage
FROM citibike_clean
GROUP BY user_type
ORDER BY round_trip_percentage DESC
""")

display(round_trips_by_user_type.toPandas())

print("""
Business Interpretation:
This result shows which user types are more likely to return bikes to the same station.
A higher round-trip percentage suggests recreational or short local usage, while a lower
percentage suggests commuting or one-way mobility patterns.
""")

,user_type,total_trips,round_trips,round_trip_percentage
0,Customer,183750,9737,5.30
1,Subscriber,1115562,17960,1.61



Business Interpretation:
This result shows which user types are more likely to return bikes to the same station.
A higher round-trip percentage suggests recreational or short local usage, while a lower
percentage suggests commuting or one-way mobility patterns.



In [22]:
# ============================================================
# Query 2(b): Most popular start stations and their usage rank
# ============================================================

popular_start_stations = spark.sql("""
SELECT
    start_station_name,
    COUNT(*) AS total_start_trips,
    DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS station_rank
FROM citibike_clean
GROUP BY start_station_name
ORDER BY station_rank
LIMIT 20
""")

display(popular_start_stations.toPandas())

print("""
Business Interpretation:
The highest-ranked start stations have the strongest demand. City planners can use
this to identify where more bikes, docks, or station expansion may be needed.
""")

26/05/23 17:42:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 1

,start_station_name,total_start_trips,station_rank
0,Pershing Square North,9761,1
1,8 Ave & W 31 St,8001,2
2,E 17 St & Broadway,7524,3
3,Broadway & E 22 St,7041,4
4,W 21 St & 6 Ave,7011,5
5,Broadway & E 14 St,6973,6
6,West St & Chambers St,6731,7
7,Broadway & W 60 St,6596,8
8,Christopher St & Greenwich St,6557,9
9,12 Ave & W 40 St,6446,10



Business Interpretation:
The highest-ranked start stations have the strongest demand. City planners can use
this to identify where more bikes, docks, or station expansion may be needed.



In [23]:
# ============================================================
# Query 2(c): Rush hours — highest demand hours
# ============================================================

rush_hours = spark.sql("""
SELECT
    HOUR(starttime) AS start_hour,
    COUNT(*) AS total_trips,
    DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS demand_rank
FROM citibike_clean
GROUP BY HOUR(starttime)
ORDER BY total_trips DESC
""")

display(rush_hours.toPandas())

print("""
Business Interpretation:
The hours with the highest demand rank represent rush hours. These peak periods help
operators plan bike redistribution, maintenance schedules, and station capacity
management around the busiest times of the day.
""")

26/05/23 17:42:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:42:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


,start_hour,total_trips,demand_rank
0,17,126578,1
1,8,115714,2
2,18,104873,3
3,16,98266,4
4,15,90935,5
5,14,88521,6
6,13,83816,7
7,9,79516,8
8,12,76324,9
9,7,70091,10



Business Interpretation:
The hours with the highest demand rank represent rush hours. These peak periods help
operators plan bike redistribution, maintenance schedules, and station capacity
management around the busiest times of the day.



In [24]:
# Confirming rider_age distribution after all cleaning steps
display(df_clean.select("rider_age").toPandas().describe())

,rider_age
count,1.299312e+06
mean,3.894920e+01
std,1.202699e+01
min,1.600000e+01
25%,2.900000e+01
50%,3.600000e+01
75%,5.000000e+01
max,1.000000e+02


In [25]:
# ============================================================
# Query 2(d): Age group feature using UDF + trip duration analysis
# ============================================================

# Age-group thresholds are derived from the dataset's own quartiles
# rather than arbitrary demographic labels:
#   25th percentile = 29  →  Young  : 16–29
#   75th percentile = 50  →  Adult  : 30–50
#                          →  Senior : 51+

def classify_age_group(age):
    if age is None:  return "Unknown"
    elif age <= 29:  return "Young"
    elif age <= 50:  return "Adult"
    else:            return "Senior"

age_group_udf = F.udf(classify_age_group, StringType())

df_clean = df_clean.withColumn("age_group", age_group_udf(F.col("rider_age")))

df_clean.createOrReplaceTempView("citibike_clean")
print("age_group column added and SQL view updated.")

age_group_duration = spark.sql("""
SELECT
    age_group,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_duration_seconds), 2) AS avg_trip_duration_seconds,
    ROUND(MIN(trip_duration_seconds), 2) AS min_trip_duration_seconds,
    ROUND(MAX(trip_duration_seconds), 2) AS max_trip_duration_seconds
FROM citibike_clean
GROUP BY age_group
ORDER BY avg_trip_duration_seconds DESC
""")

display(age_group_duration.toPandas())

print("""
Business Interpretation:
Adults (30-50) have the longest average trips at 1,013 seconds (~17 min), followed
by Young riders (16-29) at 937 seconds and Seniors (51+) at 820 seconds.
This is the opposite of the common assumption that older riders take longer trips.
Adults likely use bikes for longer commutes or mixed-purpose trips, while Seniors
make shorter local journeys possibly limited by physical endurance. Young riders
fall in between reflecting a mix of commuting and leisure. City planners can use
this to target infrastructure near areas frequented by middle-aged commuters.
""")

age_group column added and SQL view updated.


,age_group,total_trips,avg_trip_duration_seconds,min_trip_duration_seconds,max_trip_duration_seconds
0,Adult,733029,1013.00,61,2943038
1,Young,347293,937.09,61,2265874
2,Senior,218990,820.81,61,1671122



Business Interpretation:
Adults (30-50) have the longest average trips at 1,013 seconds (~17 min), followed
by Young riders (16-29) at 937 seconds and Seniors (51+) at 820 seconds.
This is the opposite of the common assumption that older riders take longer trips.
Adults likely use bikes for longer commutes or mixed-purpose trips, while Seniors
make shorter local journeys possibly limited by physical endurance. Young riders
fall in between reflecting a mix of commuting and leisure. City planners can use
this to target infrastructure near areas frequented by middle-aged commuters.



In [26]:
# ============================================================
# Query 2(e): Season feature using UDF + seasonal trip behavior
# ============================================================

def classify_season(month):
    if month is None:          return "Unknown"
    elif month in [12, 1, 2]:  return "Winter"
    elif month in [3, 4, 5]:   return "Spring"
    elif month in [6, 7, 8]:   return "Summer"
    elif month in [9, 10, 11]: return "Autumn"
    else:                      return "Unknown"

season_udf = F.udf(classify_season, StringType())
spark.udf.register("season_udf_sql", classify_season, StringType())

df_clean = df_clean.withColumn("season", season_udf(F.col("start_month")))

df_clean.createOrReplaceTempView("citibike_clean")
print("season column added, UDF registered, and SQL view updated.")

seasonal_behavior = spark.sql("""
SELECT
    season,
    COUNT(*) AS total_trips,
    COUNT(DISTINCT bike_id) AS unique_bikes_used,
    ROUND(AVG(trip_duration_seconds), 2) AS avg_duration_seconds,
    ROUND(AVG(trip_distance_km), 2) AS avg_distance_km,
    ROUND(AVG(trip_speed_kmh), 2) AS avg_speed_kmh
FROM citibike_clean
GROUP BY season
ORDER BY total_trips DESC
""")

display(seasonal_behavior.toPandas())

print("""
Business Interpretation:
Summer dominates ridership with 449,780 trips — three times more than Winter (149,897).
Summer trips are also the longest (1,018 sec), reflecting leisure riding in warm weather.
Spring riders are the fastest (9.09 km/h), suggesting mild weather drives more direct
commutes without heat fatigue. Winter has the shortest trips (831 sec) but the
second-highest speed (8.93 km/h) — only purposeful riders brave the cold, and they
ride quickly. Operationally: expand supply before Summer, schedule maintenance in
Winter, and offer promotions in Spring and Autumn to grow shoulder-season ridership.
""")

season column added, UDF registered, and SQL view updated.


,season,total_trips,unique_bikes_used,avg_duration_seconds,avg_distance_km,avg_speed_kmh
0,Summer,449780,14732,1018.31,1.85,8.56
1,Autumn,399808,16201,958.75,1.79,8.69
2,Spring,299827,14543,939.94,1.76,9.09
3,Winter,149897,15562,831.23,1.53,8.93



Business Interpretation:
Summer dominates ridership with 449,780 trips — three times more than Winter (149,897).
Summer trips are also the longest (1,018 sec), reflecting leisure riding in warm weather.
Spring riders are the fastest (9.09 km/h), suggesting mild weather drives more direct
commutes without heat fatigue. Winter has the shortest trips (831 sec) but the
second-highest speed (8.93 km/h) — only purposeful riders brave the cold, and they
ride quickly. Operationally: expand supply before Summer, schedule maintenance in
Winter, and offer promotions in Spring and Autumn to grow shoulder-season ridership.



In [27]:
# ============================================================
# Query 2(f) — Step 1: Compute per-bike usage distribution
# ============================================================

# The spec requires counting total trip seconds per bike.
# The threshold is derived dynamically as Mean + 3×Std.

bike_stats = spark.sql("""
SELECT
    bike_id,
    SUM(trip_duration_seconds) AS total_trip_seconds
FROM citibike_clean
GROUP BY bike_id
""")

stats_pd  = bike_stats.toPandas()["total_trip_seconds"].describe()
mean_secs = stats_pd["mean"]
std_secs  = stats_pd["std"]
threshold = mean_secs + 3 * std_secs

display(bike_stats.toPandas()["total_trip_seconds"].describe().to_frame())

print(f"Mean usage : {mean_secs:.2f} sec")
print(f"Std dev    : {std_secs:.2f} sec")
print(f"Threshold  : {threshold:.2f} sec  (mean + 3×std)")

,total_trip_seconds
count,1.923100e+04
mean,6.488226e+04
std,8.513760e+04
min,6.500000e+01
25%,3.537950e+04
50%,5.470500e+04
75%,7.721350e+04
max,2.964195e+06


Mean usage : 64882.26 sec
Std dev    : 85137.60 sec
Threshold  : 320295.08 sec  (mean + 3×std)


In [28]:
# ============================================================
# Query 2(f) — Step 2: Identify over-utilized bikes
# ============================================================

# Bikes whose total trip seconds exceed the threshold (mean + 3×std)
# are flagged as needing maintenance. No extra column is added —
# the HAVING clause directly filters to only the over-utilised bikes.

bike_utilization = spark.sql(f"""
SELECT
    bike_id,
    COUNT(*) AS total_trips,
    SUM(trip_duration_seconds) AS total_trip_seconds
FROM citibike_clean
GROUP BY bike_id
HAVING SUM(trip_duration_seconds) > {threshold:.2f}
ORDER BY total_trip_seconds DESC
""")

display(bike_utilization.toPandas())

print(f"Bikes needing maintenance : {bike_utilization.count():,}")
print(f"Threshold used            : {threshold:.2f} seconds  (mean + 3×std)")

print("""
Business Interpretation:
Bikes with total trip seconds above the threshold are over-utilised.
Proactive maintenance of these bikes reduces in-field breakdowns and
improves overall user experience.
""")

,bike_id,total_trips,total_trip_seconds
0,40515,29,2964195
1,25073,47,2632367
2,32225,100,2457156
3,21580,56,2313519
4,26495,66,2210383
...,...,...,...
129,29102,69,325793
130,17373,73,325506
131,35142,199,324908
132,38658,76,323340


Bikes needing maintenance : 134
Threshold used            : 320295.08 seconds  (mean + 3×std)

Business Interpretation:
Bikes with total trip seconds above the threshold are over-utilised.
Proactive maintenance of these bikes reduces in-field breakdowns and
improves overall user experience.



In [29]:
# ============================================================
# Query 2(g): Most popular end stations by user type
# ============================================================

popular_end_stations_by_user_type = spark.sql("""
WITH station_counts AS (
    SELECT
        user_type,
        end_station_name,
        COUNT(*) AS total_end_trips
    FROM citibike_clean
    GROUP BY user_type, end_station_name
),
ranked_stations AS (
    SELECT
        user_type,
        end_station_name,
        total_end_trips,
        DENSE_RANK() OVER (
            PARTITION BY user_type
            ORDER BY total_end_trips DESC
        ) AS station_rank
    FROM station_counts
)
SELECT
    user_type,
    end_station_name,
    total_end_trips,
    station_rank
FROM ranked_stations
WHERE station_rank <= 10
ORDER BY user_type, station_rank
""")

display(popular_end_stations_by_user_type.toPandas())

print("""
Business Interpretation:
Subscriber destinations tend to cluster around transit hubs and offices (commuter
pattern), while customer destinations cluster around parks and tourist landmarks
(leisure pattern).
""")

,user_type,end_station_name,total_end_trips,station_rank
0,Customer,5 Ave & E 88 St,2363,1
1,Customer,Central Park S & 6 Ave,2336,2
2,Customer,5 Ave & E 73 St,2155,3
3,Customer,Central Park West & W 72 St,2066,4
4,Customer,12 Ave & W 40 St,1984,5
5,Customer,Centre St & Chambers St,1952,6
6,Customer,West St & Chambers St,1864,7
7,Customer,Broadway & W 60 St,1672,8
8,Customer,W 34 St & 11 Ave,1658,9
9,Customer,Grand Army Plaza & Central Park S,1589,10



Business Interpretation:
Subscriber destinations tend to cluster around transit hubs and offices (commuter
pattern), while customer destinations cluster around parks and tourist landmarks
(leisure pattern).



In [30]:
# ============================================================
# Query 2(h): Station pairs with the highest trip demand
# ============================================================

station_pair_demand = spark.sql("""
WITH pair_daily_counts AS (
    SELECT
        start_station_name,
        end_station_name,
        TO_DATE(starttime) AS ride_date,
        COUNT(*) AS daily_rides
    FROM citibike_clean
    GROUP BY start_station_name, end_station_name, TO_DATE(starttime)
),
pair_summary AS (
    SELECT
        start_station_name,
        end_station_name,
        SUM(daily_rides) AS total_rides,
        ROUND(AVG(daily_rides), 2) AS avg_daily_rides
    FROM pair_daily_counts
    GROUP BY start_station_name, end_station_name
)
SELECT
    start_station_name,
    end_station_name,
    total_rides,
    avg_daily_rides,
    DENSE_RANK() OVER (ORDER BY total_rides DESC) AS demand_rank
FROM pair_summary
ORDER BY total_rides DESC
LIMIT 20
""")

display(station_pair_demand.toPandas())

print("""
Business Interpretation:
The most frequent station pairs reveal strong movement corridors. This can support
decisions about bike redistribution, route planning, and where future station
expansions may be useful.
""")

26/05/23 17:43:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:43:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:44:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:44:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 17:44:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/23 1

,start_station_name,end_station_name,total_rides,avg_daily_rides,demand_rank
0,E 7 St & Avenue A,Cooper Square & Astor Pl,585,15.00,1
1,Central Park S & 6 Ave,5 Ave & E 88 St,443,14.29,2
2,Central Park S & 6 Ave,Central Park S & 6 Ave,433,12.37,3
3,Soissons Landing,Soissons Landing,412,24.24,4
4,Yankee Ferry Terminal,Soissons Landing,411,25.69,5
5,Vesey Pl & River Terrace,North Moore St & Greenwich St,401,12.15,6
6,Soissons Landing,Picnic Point,379,23.69,7
7,Soissons Landing,Yankee Ferry Terminal,371,24.73,8
8,Picnic Point,Soissons Landing,367,22.94,9
9,12 Ave & W 40 St,West St & Chambers St,350,10.29,10



Business Interpretation:
The most frequent station pairs reveal strong movement corridors. This can support
decisions about bike redistribution, route planning, and where future station
expansions may be useful.



In [31]:
# ============================================================
# Query 2(i): Gender-based riding behavior
# ============================================================

# Gender encoding: 0 = Unknown, 1 = Male, 2 = Female

gender_behavior = spark.sql("""
SELECT
    CASE
        WHEN gender = 0 THEN 'Unknown'
        WHEN gender = 1 THEN 'Male'
        WHEN gender = 2 THEN 'Female'
        ELSE 'Other'
    END AS gender_label,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_duration_seconds), 2) AS avg_duration_seconds,
    ROUND(STDDEV(trip_duration_seconds), 2) AS std_duration_seconds,
    ROUND(AVG(trip_speed_kmh), 2) AS avg_speed_kmh,
    ROUND(STDDEV(trip_speed_kmh), 2) AS std_speed_kmh
FROM citibike_clean
GROUP BY gender
ORDER BY gender_label
""")

display(gender_behavior.toPandas())

print("""
Business Interpretation:
Male riders are faster (9.22 km/h vs 8.26 km/h) but take shorter trips (831s vs 988s)
than Female riders. This suggests males use bikes predominantly for quick point-to-point
commutes, while females take slower, longer trips more consistent with leisure or
multi-stop journeys. The Unknown gender group stands out with the longest average trip
(1,998s ~ 33 min) and lowest speed (6.42 km/h), pointing to unregistered or tourist
users who take extended, unhurried rides. The speed gap supports targeted safety
campaigns for male riders, while station placement near parks may better serve females.
""")

,gender_label,total_trips,avg_duration_seconds,std_duration_seconds,avg_speed_kmh,std_speed_kmh
0,Female,312113,988.07,7651.02,8.26,3.03
1,Male,885821,831.77,7379.20,9.22,3.32
2,Unknown,101378,1998.04,22465.36,6.42,3.56



Business Interpretation:
Male riders are faster (9.22 km/h vs 8.26 km/h) but take shorter trips (831s vs 988s)
than Female riders. This suggests males use bikes predominantly for quick point-to-point
commutes, while females take slower, longer trips more consistent with leisure or
multi-stop journeys. The Unknown gender group stands out with the longest average trip
(1,998s ~ 33 min) and lowest speed (6.42 km/h), pointing to unregistered or tourist
users who take extended, unhurried rides. The speed gap supports targeted safety
campaigns for male riders, while station placement near parks may better serve females.



In [32]:
# ============================================================
# Extra for Query 2(i): Approximate statistical significance (Welch t-test)
# ============================================================

# A large absolute t-statistic (|t| > 1.96) suggests the observed difference
# between Male and Female riders is unlikely to be due to random chance.

gender_significance = spark.sql("""
WITH gender_stats AS (
    SELECT
        gender,
        COUNT(*) AS n,
        AVG(trip_duration_seconds) AS mean_duration,
        VARIANCE(trip_duration_seconds) AS var_duration,
        AVG(trip_speed_kmh) AS mean_speed,
        VARIANCE(trip_speed_kmh) AS var_speed
    FROM citibike_clean
    WHERE gender IN (1, 2)
    GROUP BY gender
),
male   AS (SELECT * FROM gender_stats WHERE gender = 1),
female AS (SELECT * FROM gender_stats WHERE gender = 2)
SELECT
    ROUND((male.mean_duration - female.mean_duration) /
        SQRT((male.var_duration / male.n) + (female.var_duration / female.n)), 4)
        AS duration_t_statistic,

    ROUND((male.mean_speed - female.mean_speed) /
        SQRT((male.var_speed / male.n) + (female.var_speed / female.n)), 4)
        AS speed_t_statistic,

    CASE
        WHEN ABS((male.mean_duration - female.mean_duration) /
            SQRT((male.var_duration / male.n) + (female.var_duration / female.n))) > 1.96
        THEN 'Potentially significant difference'
        ELSE 'No strong evidence of significant difference'
    END AS duration_result,

    CASE
        WHEN ABS((male.mean_speed - female.mean_speed) /
            SQRT((male.var_speed / male.n) + (female.var_speed / female.n))) > 1.96
        THEN 'Potentially significant difference'
        ELSE 'No strong evidence of significant difference'
    END AS speed_result

FROM male CROSS JOIN female
""")

display(gender_significance.toPandas())

print("""
Business Interpretation:
If the t-statistic is potentially significant, the difference between male and female
riders is meaningful and not just noise. This supports targeted policy or service
decisions based on gender riding patterns.
""")

,duration_t_statistic,speed_t_statistic,duration_result,speed_result
0,-9.9046,147.6095,Potentially significant difference,Potentially significant difference



Business Interpretation:
If the t-statistic is potentially significant, the difference between male and female
riders is meaningful and not just noise. This supports targeted policy or service
decisions based on gender riding patterns.



In [33]:
# ============================================================
# Query 2(j): Weekday vs weekend trip behavior
# ============================================================

# Spark dayofweek: 1 = Sunday, 7 = Saturday → weekends are 1 and 7.

weekday_weekend_behavior = spark.sql("""
SELECT
    CASE
        WHEN DAYOFWEEK(starttime) IN (1, 7) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_duration_seconds), 2) AS avg_duration_seconds,
    ROUND(AVG(trip_distance_km), 2) AS avg_distance_km,
    ROUND(AVG(trip_speed_kmh), 2) AS avg_speed_kmh
FROM citibike_clean
GROUP BY
    CASE
        WHEN DAYOFWEEK(starttime) IN (1, 7) THEN 'Weekend'
        ELSE 'Weekday'
    END
ORDER BY total_trips DESC
""")

display(weekday_weekend_behavior.toPandas())

print("""
Business Interpretation:
Weekday rides are shorter and faster (commute pattern). Weekend rides are longer
and slower (leisure pattern). Fleet rebalancing strategy should differ between
the two day types.
""")

,day_type,total_trips,avg_duration_seconds,avg_distance_km,avg_speed_kmh
0,Weekday,973745,892.57,1.76,9.05
1,Weekend,325567,1162.93,1.81,7.92



Business Interpretation:
Weekday rides are shorter and faster (commute pattern). Weekend rides are longer
and slower (leisure pattern). Fleet rebalancing strategy should differ between
the two day types.



---
# Part 3 — SparkML: Predict Rider Gender

In [34]:
from pyspark.ml.feature        import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation     import MulticlassClassificationEvaluator
from pyspark.ml                import Pipeline

print("SparkML imports loaded.")

SparkML imports loaded.


In [35]:
# ============================================================
# Prepare ML dataset
# ============================================================

# Features chosen based on availability and relevance to gender prediction:
#   - rider_age        : demographic indicator
#   - trip_duration_seconds : behavioural pattern
#   - trip_distance_km : behavioural pattern
#   - trip_speed_kmh   : behavioural pattern
#   - start_month      : seasonal usage pattern
#
# Gender 0 (Unknown) is excluded — it carries no label signal for training.
# dropna() removes any rows with NULL in the selected feature columns.

feature_cols = [
    "rider_age",
    "trip_duration_seconds",
    "trip_distance_km",
    "trip_speed_kmh",
    "start_month"
]

ml_df = df_clean \
    .filter(F.col("gender").isin(1, 2)) \
    .select(feature_cols + ["gender"]) \
    .dropna() \
    .withColumn("label", (F.col("gender") - 1).cast("double"))
# label: Male=0, Female=1

label_dist = {r["label"]: r["count"] for r in ml_df.groupBy("label").count().collect()}
total_ml   = sum(label_dist.values())

print("ML dataset ready:")
print(f"  Total rows  : {total_ml:,}")
print(f"  Male  (0)   : {label_dist.get(0.0, 0):,}  ({label_dist.get(0.0,0)/total_ml*100:.1f}%)")
print(f"  Female (1)  : {label_dist.get(1.0, 0):,}  ({label_dist.get(1.0,0)/total_ml*100:.1f}%)")
print(f"  Features    : {feature_cols}")

ML dataset ready:
  Total rows  : 1,197,934
  Male  (0)   : 885,821  (73.9%)
  Female (1)  : 312,113  (26.1%)
  Features    : ['rider_age', 'trip_duration_seconds', 'trip_distance_km', 'trip_speed_kmh', 'start_month']


In [36]:
# ============================================================
# Assemble features, scale, and split
# ============================================================

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("Train / Test split:")
print(f"  Train : {train_df.count():,} rows")
print(f"  Test  : {test_df.count():,} rows")

Train / Test split:


  Train : 958,057 rows


  Test  : 239,877 rows


In [37]:
# ============================================================
# Model 1 — Logistic Regression
# ============================================================

print("Training Logistic Regression...")

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model    = lr_pipeline.fit(train_df)
lr_preds    = lr_model.transform(test_df)

lr_acc = evaluator.evaluate(lr_preds)
lr_f1  = f1_evaluator.evaluate(lr_preds)

print(f"Accuracy : {lr_acc*100:.2f}%")
print(f"F1 Score : {lr_f1*100:.2f}%")

Training Logistic Regression...


Accuracy : 73.87%
F1 Score : 62.81%


In [38]:
# ============================================================
# Model 2 — Decision Tree
# ============================================================

print("Training Decision Tree...")

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=10,
    seed=42
)

dt_pipeline = Pipeline(stages=[assembler, scaler, dt])
dt_model    = dt_pipeline.fit(train_df)
dt_preds    = dt_model.transform(test_df)

dt_acc = evaluator.evaluate(dt_preds)
dt_f1  = f1_evaluator.evaluate(dt_preds)

print(f"Accuracy : {dt_acc*100:.2f}%")
print(f"F1 Score : {dt_f1*100:.2f}%")

print("\nFeature Importances (Decision Tree):")
print("─" * 40)
for feat, imp in sorted(
    zip(feature_cols, dt_model.stages[-1].featureImportances.toArray()),
    key=lambda x: -x[1]
):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<25}: {imp:.4f}  {bar}")

Training Decision Tree...


Accuracy : 73.85%
F1 Score : 62.85%

Feature Importances (Decision Tree):
────────────────────────────────────────
  trip_speed_kmh           : 0.6461  █████████████████████████
  trip_duration_seconds    : 0.1672  ██████
  rider_age                : 0.1474  █████
  start_month              : 0.0213  
  trip_distance_km         : 0.0180  


In [39]:
# ============================================================
# Model 3 — Random Forest
# ============================================================

print("Training Random Forest (may take a minute)...")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])
rf_model    = rf_pipeline.fit(train_df)
rf_preds    = rf_model.transform(test_df)

rf_acc = evaluator.evaluate(rf_preds)
rf_f1  = f1_evaluator.evaluate(rf_preds)

print(f"Accuracy : {rf_acc*100:.2f}%")
print(f"F1 Score : {rf_f1*100:.2f}%")

print("\nFeature Importances (Random Forest):")
print("─" * 40)
for feat, imp in sorted(
    zip(feature_cols, rf_model.stages[-1].featureImportances.toArray()),
    key=lambda x: -x[1]
):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<25}: {imp:.4f}  {bar}")

Training Random Forest (may take a minute)...


26/05/23 17:48:15 WARN DAGScheduler: Broadcasting large task binary with size 1001.7 KiB
26/05/23 17:48:25 WARN DAGScheduler: Broadcasting large task binary with size 1888.2 KiB
26/05/23 17:48:38 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/05/23 17:48:54 WARN DAGScheduler: Broadcasting large task binary with size 1130.9 KiB
26/05/23 17:48:55 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/05/23 17:49:14 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/05/23 17:49:17 WARN DAGScheduler: Broadcasting large task binary with size 13.7 MiB
26/05/23 17:49:42 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/05/23 17:49:44 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/05/23 17:49:56 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


Accuracy : 73.88%
F1 Score : 62.78%

Feature Importances (Random Forest):
────────────────────────────────────────
  trip_speed_kmh           : 0.5864  ███████████████████████
  trip_duration_seconds    : 0.1901  ███████
  rider_age                : 0.1369  █████
  trip_distance_km         : 0.0585  ██
  start_month              : 0.0281  █


In [40]:
# ============================================================
# Model Comparison Summary
# ============================================================

results = {
    "Logistic Regression" : (lr_acc, lr_f1),
    "Decision Tree"       : (dt_acc, dt_f1),
    "Random Forest"       : (rf_acc, rf_f1),
}

best = max(results, key=lambda k: results[k][0])

print("╔══════════════════════════════════════════════════════╗")
print("║             MODEL COMPARISON SUMMARY                ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"  {'Model':<22}  {'Accuracy':>9}  {'F1 Score':>9}")
print("─" * 54)
for name, (acc, f1) in sorted(results.items(), key=lambda x: -x[1][0]):
    tag = "  <- BEST" if name == best else ""
    print(f"  {name:<22}  {acc*100:>8.2f}%  {f1*100:>8.2f}%{tag}")
print("╚══════════════════════════════════════════════════════╝")

print("""
Discussion:
- All three models perform nearly identically: RF 73.88%, LR 73.87%, DT 73.85%.
  The margin is under 0.03% so no model has a meaningful accuracy edge on this task.
- Random Forest is chosen as best due to its ensemble nature reducing variance,
  even though the accuracy advantage over the others is negligible.
- Logistic Regression matches Random Forest almost exactly, suggesting the gender
  signal in this dataset is largely linear rather than deeply non-linear.
- The most important feature by far is trip_speed_kmh (64.6% importance in the
  Decision Tree), followed by trip_duration_seconds (16.7%) and rider_age (14.7%).
  start_month (2.1%) and trip_distance_km (1.8%) contribute very little.
  The dominance of trip_speed_kmh confirms the finding from Query 2(i): male riders
  are consistently faster, making speed the clearest gender signal in the data.
- F1 (~62.8%) is notably lower than accuracy (~73.9%) because the dataset is heavily
  imbalanced (73.9% Male vs 26.1% Female). Models default toward predicting Male,
  inflating accuracy while underperforming on Female prediction.
""")

╔══════════════════════════════════════════════════════╗
║             MODEL COMPARISON SUMMARY                ║
╠══════════════════════════════════════════════════════╣
  Model                    Accuracy   F1 Score
──────────────────────────────────────────────────────
  Random Forest              73.88%     62.78%  <- BEST
  Logistic Regression        73.87%     62.81%
  Decision Tree              73.85%     62.85%
╚══════════════════════════════════════════════════════╝

Discussion:
- All three models perform nearly identically: RF 73.88%, LR 73.87%, DT 73.85%.
  The margin is under 0.03% so no model has a meaningful accuracy edge on this task.
- Random Forest is chosen as best due to its ensemble nature reducing variance,
  even though the accuracy advantage over the others is negligible.
- Logistic Regression matches Random Forest almost exactly, suggesting the gender
  signal in this dataset is largely linear rather than deeply non-linear.
- The most important feature by far is 

---
# Bonus — Interactive Bokeh Dashboard

In [41]:
!pip install bokeh --quiet

In [42]:
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.models   import (ColumnDataSource, HoverTool, Select, CustomJS, Div)
from bokeh.layouts  import column, row

output_notebook()

# ── Extra Spark queries — per user type data for all 4 charts ─────────

# Chart 1: hourly demand per user type
rush_all_pd = rush_hours.toPandas().sort_values("start_hour")
rush_sub_pd = spark.sql("""
    SELECT HOUR(starttime) AS start_hour, COUNT(*) AS total_trips
    FROM citibike_clean WHERE user_type = 'Subscriber'
    GROUP BY HOUR(starttime) ORDER BY start_hour
""").toPandas().sort_values("start_hour")
rush_cus_pd = spark.sql("""
    SELECT HOUR(starttime) AS start_hour, COUNT(*) AS total_trips
    FROM citibike_clean WHERE user_type = 'Customer'
    GROUP BY HOUR(starttime) ORDER BY start_hour
""").toPandas().sort_values("start_hour")

# Chart 2: top 10 end stations per user type
end_all_pd = spark.sql("""
    SELECT end_station_name, COUNT(*) AS total_end_trips
    FROM citibike_clean
    GROUP BY end_station_name ORDER BY total_end_trips DESC LIMIT 10
""").toPandas().sort_values("total_end_trips")

end_sub_pd = (popular_end_stations_by_user_type
              .filter(F.col("user_type") == "Subscriber")
              .filter(F.col("station_rank") <= 10)
              .orderBy("total_end_trips")
              .toPandas())
end_cus_pd = (popular_end_stations_by_user_type
              .filter(F.col("user_type") == "Customer")
              .filter(F.col("station_rank") <= 10)
              .orderBy("total_end_trips")
              .toPandas())

# Chart 3: gender avg speed per user type
def prep_gender(df):
    df = df[df["gender_label"] != "Unknown"].copy()
    df["color"] = df["gender_label"].map({"Male": "#4e91d2", "Female": "#e07b7b"})
    return df.sort_values("gender_label")

gender_all_pd = prep_gender(gender_behavior.toPandas())
gender_sub_pd = prep_gender(spark.sql("""
    SELECT CASE WHEN gender=1 THEN 'Male' WHEN gender=2 THEN 'Female' END AS gender_label,
           ROUND(AVG(trip_speed_kmh), 2) AS avg_speed_kmh
    FROM citibike_clean WHERE gender IN (1,2) AND user_type = 'Subscriber'
    GROUP BY gender
""").toPandas())
gender_cus_pd = prep_gender(spark.sql("""
    SELECT CASE WHEN gender=1 THEN 'Male' WHEN gender=2 THEN 'Female' END AS gender_label,
           ROUND(AVG(trip_speed_kmh), 2) AS avg_speed_kmh
    FROM citibike_clean WHERE gender IN (1,2) AND user_type = 'Customer'
    GROUP BY gender
""").toPandas())

# Chart 4: top 15 most-used bikes per user type
bikes_all_pd = bike_utilization.limit(15).toPandas().sort_values("total_trip_seconds")
bikes_sub_pd = spark.sql("""
    SELECT bike_id, COUNT(*) AS total_trips,
           SUM(trip_duration_seconds) AS total_trip_seconds
    FROM citibike_clean WHERE user_type = 'Subscriber'
    GROUP BY bike_id ORDER BY total_trip_seconds DESC
""").limit(15).toPandas().sort_values("total_trip_seconds")
bikes_cus_pd = spark.sql("""
    SELECT bike_id, COUNT(*) AS total_trips,
           SUM(trip_duration_seconds) AS total_trip_seconds
    FROM citibike_clean WHERE user_type = 'Customer'
    GROUP BY bike_id ORDER BY total_trip_seconds DESC
""").limit(15).toPandas().sort_values("total_trip_seconds")

for df in [bikes_all_pd, bikes_sub_pd, bikes_cus_pd]:
    df["bike_id_str"] = df["bike_id"].astype(str)

print("All data collected.")

# ── Convert DataFrames to plain dicts for CustomJS ────────────────────

def rush_d(df):
    return {"start_hour": df["start_hour"].tolist(),
            "total_trips": df["total_trips"].tolist()}

def station_d(df):
    return {"end_station_name": df["end_station_name"].tolist(),
            "total_end_trips":  df["total_end_trips"].tolist()}

def gender_d(df):
    return {"gender_label": df["gender_label"].tolist(),
            "value":        df["avg_speed_kmh"].tolist(),
            "color":        df["color"].tolist()}

def bikes_d(df):
    return {"bike_id_str":        df["bike_id_str"].tolist(),
            "total_trip_seconds": df["total_trip_seconds"].tolist(),
            "total_trips":        df["total_trips"].tolist()}

# ── Bokeh sources (start with "All") ──────────────────────────────────

src_rush     = ColumnDataSource(rush_d(rush_all_pd))
src_stations = ColumnDataSource(station_d(end_all_pd))
src_gender   = ColumnDataSource(gender_d(gender_all_pd))
src_bikes    = ColumnDataSource(bikes_d(bikes_all_pd))

# ── Charts ────────────────────────────────────────────────────────────

# Chart 1 — Hourly demand
p_hourly = figure(title="Hourly Trip Demand",
                  x_axis_label="Hour of Day", y_axis_label="Total Trips",
                  width=460, height=300, tools="hover,pan,reset")
p_hourly.line("start_hour",   "total_trips", source=src_rush,
              line_width=2.5, color="#378ADD")
p_hourly.circle("start_hour", "total_trips", source=src_rush,
                size=7, color="#378ADD")
p_hourly.select_one(HoverTool).tooltips = [
    ("Hour",  "@start_hour:00"),
    ("Trips", "@total_trips{,}")
]

# Chart 2 — End stations
p_stations = figure(
    y_range=end_all_pd["end_station_name"].tolist(),
    title="Top 10 End Stations",
    x_axis_label="Total Trips",
    width=460, height=300, tools="hover,reset"
)
p_stations.hbar(y="end_station_name", right="total_end_trips",
                source=src_stations, height=0.6, color="#D85A30", alpha=0.85)
p_stations.add_tools(HoverTool(tooltips=[
    ("Station", "@end_station_name"),
    ("Trips",   "@total_end_trips{,}")
]))

# Chart 3 — Gender avg speed
p_gender = figure(
    x_range=gender_all_pd["gender_label"].tolist(),
    title="Gender Avg Speed (km/h)",
    x_axis_label="Gender", y_axis_label="Avg Speed (km/h)",
    width=460, height=300, tools="hover,reset"
)
p_gender.vbar(x="gender_label", top="value", source=src_gender,
              width=0.45, color="color", alpha=0.85)
p_gender.add_tools(HoverTool(tooltips=[
    ("Gender",    "@gender_label"),
    ("Avg Speed", "@value{0.00} km/h")
]))

# Chart 4 — Bike utilization
p_bikes = figure(
    y_range=bikes_all_pd["bike_id_str"].tolist(),
    title="Top Over-Utilised Bikes",
    x_axis_label="Total Trip Seconds",
    width=460, height=300, tools="hover,reset"
)
p_bikes.hbar(y="bike_id_str", right="total_trip_seconds",
             source=src_bikes, height=0.6, color="#E24B4A", alpha=0.85)
p_bikes.add_tools(HoverTool(tooltips=[
    ("Bike ID",   "@bike_id_str"),
    ("Total Sec", "@total_trip_seconds{,}"),
    ("Trips",     "@total_trips{,}")
]))

# ── Single filter — drives all 4 charts at once ───────────────────────

user_type_sel = Select(
    title="Filter by User Type:",
    value="All",
    options=["All", "Subscriber", "Customer"],
    width=250
)

cb = CustomJS(
    args=dict(
        src_rush=src_rush, src_stations=src_stations,
        src_gender=src_gender, src_bikes=src_bikes,
        yr_st=p_stations.y_range, yr_bk=p_bikes.y_range,
        all_r=rush_d(rush_all_pd), sub_r=rush_d(rush_sub_pd), cus_r=rush_d(rush_cus_pd),
        all_s=station_d(end_all_pd), sub_s=station_d(end_sub_pd), cus_s=station_d(end_cus_pd),
        all_g=gender_d(gender_all_pd), sub_g=gender_d(gender_sub_pd), cus_g=gender_d(gender_cus_pd),
        all_b=bikes_d(bikes_all_pd), sub_b=bikes_d(bikes_sub_pd), cus_b=bikes_d(bikes_cus_pd),
    ),
    code="""
        const v = cb_obj.value;
        const r = v==='Subscriber' ? sub_r : v==='Customer' ? cus_r : all_r;
        const s = v==='Subscriber' ? sub_s : v==='Customer' ? cus_s : all_s;
        const g = v==='Subscriber' ? sub_g : v==='Customer' ? cus_g : all_g;
        const b = v==='Subscriber' ? sub_b : v==='Customer' ? cus_b : all_b;

        src_rush.data     = r;
        src_stations.data = s;
        src_gender.data   = g;
        src_bikes.data    = b;

        yr_st.factors = s['end_station_name'];
        yr_bk.factors = b['bike_id_str'];

        src_rush.change.emit();
        src_stations.change.emit();
        src_gender.change.emit();
        src_bikes.change.emit();
    """
)
user_type_sel.js_on_change("value", cb)

# ── Layout — 2×2 grid with filter on top ─────────────────────────────

dashboard = column(
    Div(text="<h2 style='margin:4px 0'>Citi Bike Dashboard</h2>"),
    user_type_sel,
    row(p_hourly,  p_stations),
    row(p_gender,  p_bikes)
)

show(dashboard)
print("Dashboard rendered.")
print("  User Type filter drives all 4 charts simultaneously.")

Loading BokehJS ...

All data collected.


Dashboard rendered.
  User Type filter drives all 4 charts simultaneously.
